In [1]:
from pathlib import Path

DATA_RAW = Path("datasets/raw_old")
IMAGES_DIR = "images"
LABELS_DIR = "labels"
SPLITS = ["train", "val", "test"]


# class mapping
CLASS_NAMES = {
    0: "no_animal",
    1: "red_deer",
    2: "roe_deer",
    3: "chamois",
    4: "human",
    5: "alpine_ibex",
    6: "fallow_deer",
    7: "unknown",
    8: "dog",
    9: "bird",
    10: "wild_boar",
    11: "hybrid_pig"
}

In [2]:
# Imagesize
from PIL import Image
import os

img_path = DATA_RAW / "images" / "train"

for f in os.listdir(img_path):
    if f.endswith((".jpg")):
        img = Image.open(os.path.join(img_path, f))
        print(f"name: {f}, size (w/h): {img.size}")
        break

name: 0_8082.jpg, size (w/h): (2048, 2048)


In [3]:
from pathlib import Path
from collections import Counter
import numpy as np
import csv

images_total = 0
images_with_animals = 0
images_without_animals = 0

animals_per_image = []

class_counter = Counter()

In [4]:
for split in SPLITS:
    label_dir = DATA_RAW / LABELS_DIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        images_total += 1

        with open(file, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # Case: empty or only "0" → no animal image
        if len(lines) == 0 or (len(lines) == 1 and lines[0].split()[0] == "0"):
            images_without_animals += 1
            animals_per_image.append(0)
            continue

        # image contains animals
        images_with_animals += 1
        animals_per_image.append(len(lines))

        for line in lines:
            cls = int(line.split()[0])


            class_counter[cls] += 1

total_animals = sum(class_counter.values())
avg_animals_per_image = total_animals / images_total if images_total else 0

animals_array = np.array(animals_per_image)

In [5]:
print(f"Total images: {images_total}")
print(f"Images with animals: {images_with_animals}")
print(f"Images without animals: {images_without_animals}")
print(f"Percentage with animals: {images_with_animals / images_total * 100:.2f}%")

print("\n--- Animal stats ---")
print(f"Total animals: {total_animals}")
print(f"Average animals per image: {avg_animals_per_image:.2f}")
print(f"Max animals in one image: {animals_array.max()}")

print(f"95th percentile animals/image: {np.percentile(animals_array, 95):.2f}")
print(f"99th percentile animals/image: {np.percentile(animals_array, 99):.2f}")

print("\n--- Per class distribution ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    name = CLASS_NAMES[cls_id]
    count = class_counter.get(cls_id, 0)

    print(f"{cls_id:2d} {name:15s}: {count}")


Total images: 19084
Images with animals: 18354
Images without animals: 730
Percentage with animals: 96.17%

--- Animal stats ---
Total animals: 70922
Average animals per image: 3.72
Max animals in one image: 69
95th percentile animals/image: 12.00
99th percentile animals/image: 36.00

--- Per class distribution ---
 0 no_animal      : 1647
 1 red_deer       : 27696
 2 roe_deer       : 3615
 3 chamois        : 5111
 4 human          : 23686
 5 alpine_ibex    : 3355
 6 fallow_deer    : 3779
 7 unknown        : 1187
 8 dog            : 846
 9 bird           : 0
10 wild_boar      : 0
11 hybrid_pig     : 0


### Class occurrences per split

Compare box counts per class across train / val / test. A large mismatch between splits (e.g. a class only in val) will hurt training and make mAP misleading.

In [ ]:
IMAGE_SIZE = 1024


def collect_box_sizes(label_dir):
    widths_px, heights_px = [], []
    widths_norm, heights_norm = [], []

    for file in label_dir.glob("*.txt"):
        for line in file.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            w, h = map(float, parts[3:5])
            widths_norm.append(w)
            heights_norm.append(h)
            widths_px.append(w * IMAGE_SIZE)
            heights_px.append(h * IMAGE_SIZE)

    return widths_px, heights_px, widths_norm, heights_norm


def print_size_stats(name, w_px, h_px, w_norm, h_norm):
    if not w_px:
        print(f"{name}: no boxes")
        return

    w_px, h_px = np.array(w_px), np.array(h_px)
    w_norm, h_norm = np.array(w_norm), np.array(h_norm)

    print(f"\n{name} ({len(w_px)} boxes)")
    print(
        f"  width  | min={w_px.min():6.1f}px ({w_norm.min:.4f})  "
        f"avg={w_px.mean():6.1f}px ({w_norm.mean:.4f})  "
        f"max={w_px.max():6.1f}px ({w_norm.max:.4f})"
    )
    print(
        f"  height | min={h_px.min():6.1f}px ({h_norm.min:.4f})  "
        f"avg={h_px.mean():6.1f}px ({h_norm.mean:.4f})  "
        f"max={h_px.max():6.1f}px ({h_norm.max:.4f})"
    )
    min_side = np.minimum(w_px, h_px)
    print(
        f"  min side (min(w,h)) | min={min_side.min():6.1f}px  "
        f"avg={min_side.mean():6.1f}px  max={min_side.max():6.1f}px"
    )


all_w_px, all_h_px, all_w_n, all_h_n = [], [], [], []

print("--- Box size stats per split ---")
for split in SPLITS:
    label_dir = DATA_RAW / LABELS_DIR / split
    if not label_dir.exists():
        continue

    w_px, h_px, w_n, h_n = collect_box_sizes(label_dir)
    print_size_stats(split.upper(), w_px, h_px, w_n, h_n)

    all_w_px.extend(w_px)
    all_h_px.extend(h_px)
    all_w_n.extend(w_n)
    all_h_n.extend(h_n)

print("\n--- Box size stats (all splits) ---")
print_size_stats("ALL", all_w_px, all_h_px, all_w_n, all_h_n)

In [ ]:
class_counter_per_split = {split: Counter() for split in SPLITS}

for split in SPLITS:
    label_dir = DATA_RAW / LABELS_DIR / split
    if not label_dir.exists():
        continue

    for file in label_dir.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls = int(line.split()[0])
                class_counter_per_split[split][cls] += 1

# table: box counts per class per split
print("--- Per-class occurrences per split (box counts) ---\n")
col_w = 10
header = f"{'id':>3} {'class':15s}" + "".join(f"{s:>{col_w}s}" for s in SPLITS)
print(header)
print("-" * len(header))

for cls_id in sorted(CLASS_NAMES.keys()):
    counts = [class_counter_per_split[s].get(cls_id, 0) for s in SPLITS]
    if sum(counts) == 0:
        continue
    print(
        f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
        + "".join(f"{c:>{col_w}d}" for c in counts)
    )

print("-" * len(header))
split_totals = [sum(class_counter_per_split[s].values()) for s in SPLITS]
print(f"{'':3} {'TOTAL':15s}" + "".join(f"{t:>{col_w}d}" for t in split_totals))

# percentage within each split
print("\n--- Per-class share within each split (%) ---\n")
print(header)
print("-" * len(header))

for cls_id in sorted(CLASS_NAMES.keys()):
    pcts = []
    has_any = False
    for split in SPLITS:
        total = split_totals[SPLITS.index(split)] or 1
        count = class_counter_per_split[split].get(cls_id, 0)
        pcts.append(count / total * 100)
        has_any |= count > 0
    if not has_any:
        continue
    print(
        f"{cls_id:3d} {CLASS_NAMES[cls_id]:15s}"
        + "".join(f"{p:>{col_w}.1f}" for p in pcts)
    )

--- Per-class occurrences per split (box counts) ---

 id class               train       val      test
-------------------------------------------------
  0 no_animal            1557         0         0
  1 red_deer            16356       414      2055
  2 roe_deer              284         0       949
  3 chamois               515         0         0
  4 human                3637         0         4
  5 alpine_ibex          1062         0         0
  6 fallow_deer          1260         0      1540
  7 unknown                 0         0       600
-------------------------------------------------
    TOTAL               24671       414      5148

--- Per-class share within each split (%) ---

 id class               train       val      test
-------------------------------------------------
  0 no_animal             6.3       0.0       0.0
  1 red_deer             66.3     100.0      39.9
  2 roe_deer              1.2       0.0      18.4
  3 chamois               2.1       0.0       0.

In [11]:
print("\n--- Class imbalance (percentage, FULL) ---")

all_class_ids = sorted(CLASS_NAMES.keys())

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    print(f"{CLASS_NAMES[cls_id]:15s}: {pct:.2f}%")


--- Class imbalance (percentage, FULL) ---
no_animal      : 4.45%
red_deer       : 62.72%
roe_deer       : 4.11%
chamois        : 1.72%
human          : 12.13%
alpine_ibex    : 3.54%
fallow_deer    : 9.33%
unknown        : 2.00%
dog            : 0.00%
bird           : 0.00%
wild_boar      : 0.00%
hybrid_pig     : 0.00%


In [12]:
print("\n--- Rare classes (<5%) ---")

for cls_id in all_class_ids:
    count = class_counter.get(cls_id, 0)
    pct = (count / total_animals) * 100 if total_animals else 0

    if pct < 5:
        print(f"{CLASS_NAMES[cls_id]:15s}: {count:5d} - {pct:.2f}%")


--- Rare classes (<5%) ---
no_animal      :  1336 - 4.45%
roe_deer       :  1233 - 4.11%
chamois        :   515 - 1.72%
alpine_ibex    :  1062 - 3.54%
unknown        :   600 - 2.00%
dog            :     0 - 0.00%
bird           :     0 - 0.00%
wild_boar      :     0 - 0.00%
hybrid_pig     :     0 - 0.00%


In [13]:
from collections import Counter as C
dist = C(animals_per_image)

print("--- Animals per image distribution ---")
for k in sorted(dist):
    print(f"{k} animals: {dist[k]} images")

--- Animals per image distribution ---
0 animals: 221 images
1 animals: 5692 images
2 animals: 2721 images
3 animals: 1291 images
4 animals: 607 images
5 animals: 437 images
6 animals: 269 images
7 animals: 259 images
8 animals: 114 images
9 animals: 136 images
10 animals: 61 images
11 animals: 48 images
12 animals: 68 images
13 animals: 19 images
14 animals: 12 images
15 animals: 17 images
16 animals: 28 images
17 animals: 11 images
18 animals: 16 images
19 animals: 13 images
20 animals: 4 images
21 animals: 3 images
22 animals: 3 images
23 animals: 12 images
24 animals: 4 images
25 animals: 14 images
26 animals: 4 images


In [14]:
with open("analysis/dataset_class_distribution.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["class_id", "class_name", "count", "percentage"])

    for cls_id, count in sorted(class_counter.items()):
        pct = (count / total_animals) * 100 if total_animals else 0
        writer.writerow([cls_id, CLASS_NAMES.get(cls_id), count, pct])

print("\nCSV saved: dataset_class_distribution.csv")


CSV saved: dataset_class_distribution.csv


In [15]:
# look at black and blurred images -> count:
from utils.preprocess_methods import is_mostly_black, is_blurry

from pathlib import Path

def scan_quality_stats(base_path: Path, splits, images_dir="images"):
    stats = {}

    for split in splits:
        img_dir = base_path / images_dir / split

        total = 0
        black = 0
        blurry = 0

        for img_path in img_dir.glob("*.jpg"):
            total += 1

            if is_mostly_black(img_path):
                black += 1
                continue

            if is_blurry(img_path):
                blurry += 1
                continue

        stats[split] = {
            "total": total,
            "black": black,
            "blurry": blurry,
            "kept": total - black - blurry
        }

    return stats

In [16]:

from pathlib import Path

def scan_quality_stats(base_path: Path, splits, images_dir="images"):
    stats = {}

    for split in splits:
        img_dir = base_path / images_dir / split

        total = 0
        black = 0
        blurry = 0

        for img_path in img_dir.glob("*.jpg"):
            total += 1

            if is_mostly_black(img_path):
                black += 1
                continue

            if is_blurry(img_path):
                blurry += 1
                continue

        stats[split] = {
            "total": total,
            "black": black,
            "blurry": blurry,
            "kept": total - black - blurry
        }

    return stats

In [17]:
raw_stats = scan_quality_stats(DATA_RAW, SPLITS)

print("\n===== QUALITY CHECK (RAW DATA) =====")
for split, s in raw_stats.items():
    print(
        f"{split.upper():5s} | "
        f"total={s['total']:5d} | "
        f"kept={s['kept']:5d} | "
        f"black={s['black']:5d} | "
        f"blurry={s['blurry']:5d}"
    )

KeyboardInterrupt: 